# Project 10: Image Classification — ANN vs CNN

**Dataset:** Fashion-MNIST (Zalando, 10,000 samples, 28x28 grayscale, 10 clothing classes)

**Goal:** Compare a traditional Artificial Neural Network (ANN) with a Convolutional Neural Network (CNN) concept. Understand why CNNs are better for images.

In [ ]:
# ====== Setup ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    df = pd.read_csv(f'{DATA_URL}/{fn}')
    print(f'Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols')
    return df
print('Setup OK!')
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 1. Load Fashion-MNIST

In [ ]:
# Fashion-MNIST: Zalando product images — more realistic than handwritten digits!
print('Loading Fashion-MNIST (10 clothing classes)...')
data = fetch_openml('Fashion-MNIST', version=1, parser='auto')
X = data.data.values.astype(np.float32) / 255.0  # Normalize to [0,1]
y = data.target.values.astype(int)

class_names = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

# Use 10,000 samples for speed
np.random.seed(42)
idx = np.random.choice(len(X), 10000, replace=False)
X, y = X[idx], y[idx]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Samples: {len(X):,}, Pixels: {X.shape[1]} (28x28 = 784)')
print(f'Train: {X_train.shape[0]:,}, Test: {X_test.shape[0]:,}')

## 2. Sample Images & ANN Training

In [ ]:
# Show one sample per class — 每个类别显示一个样本
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    r, c = i // 5, i % 5
    idx = np.where(y_train == i)[0][0]
    axes[r,c].imshow(X_train[idx].reshape(28,28), cmap='gray')
    axes[r,c].set_title(class_names[i]); axes[r,c].axis('off')
plt.suptitle('Fashion-MNIST: One Sample per Class', fontsize=14)
plt.tight_layout(); plt.show()

# Train ANN — 训练人工神经网络
# Architecture: 784 (flattened 28x28) -> 128 (ReLU) -> 64 (ReLU) -> 10 (Softmax)
# Total parameters: ~109K
ann = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
                    max_iter=50, random_state=42)
ann.fit(X_train, y_train)
y_pred = ann.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'ANN Test Accuracy: {acc:.4f}')
print(f'Total parameters: ~109K')

plt.figure(figsize=(10, 5))
plt.plot(ann.loss_curve_, linewidth=2, color='#2e86de')
plt.xlabel('Iteration'); plt.ylabel('Loss'); plt.title('ANN Training Loss')
plt.grid(True, alpha=0.3); plt.show()

## 3. ANN Analysis & CNN Comparison

In [ ]:
# Confusion matrix — 混淆矩阵fig, ax = plt.subplots(figsize=(10, 8))ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues',    display_labels=class_names, xticks_rotation=45)ax.set_title(f'ANN Confusion Matrix (Accuracy={acc:.3f})')plt.tight_layout(); plt.show()# Show misclassifications — 展示错误分类mis = np.where(y_pred != y_test)[0]print(f'Total misclassified: {len(mis)} / {len(y_test)} ({len(mis)/len(y_test):.1%})')print('\nSample misclassifications:')for i in mis[:8]:    t = y_test.iloc[i] if hasattr(y_test,'iloc') else y_test[i]    p = y_pred[i]    print(f'  True: {class_names[t]:12s} -> Predicted: {class_names[p]:12s}')print('\n' + '=' * 60)print('ANN vs CNN — Why Convolution?')print('=' * 60)print()print('ANN:')print('  Input:   784 (flattened 28x28) - LOSES spatial structure')print('  Params:  ~109K')print('  Problem: if a shirt shifts 5 pixels, ANN sees completely')print('           different values! No translation invariance.')print()print('CNN:')print('  Input:   28x28x1 - PRESERVES 2D structure')print('  Params:  ~25K (4x FEWER than ANN!)')print('  Trick:   Conv filters slide across the image')print('           Same filter everywhere = weight sharing')print('           Pooling = keeps only strongest features')print()print('Result: CNN is MORE accurate with FEWER parameters.')print('        Because it uses the structure of the data!')

## Check Your Understanding
1. The ANN flattens 28x28 into 784 pixels. What spatial information is lost? (Hint: what if the shirt is slightly rotated?)
2. How can CNN have 4x fewer parameters but higher accuracy? What's the "trick"?
3. Look at the confusion matrix — which classes are most confused with each other? Why would this happen?
4. If you were deploying a model to detect tumors in X-ray images, would you choose ANN or CNN? Justify your answer.
5. (Challenge) How would you improve the ANN's accuracy? (Hint: more layers? more data? data augmentation?)